# Logits Preprocessing and Data Engineering

In [1]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/datax/code_smells/generation/dataset',
            #['greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling']
            'decoding_strategy': 'greedy_search',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs/generation', 
        'output_path' : '/workspaces/CodeSmells/datax/code_smells/logits/generation',
        'callbacks_path' : '/workspaces/CodeSmells/datax/code_smells/callbacks/generation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [3]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [4]:
from transformers import CodeLlamaTokenizer, LlamaForCausalLM
from datasets import load_dataset

In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['decoding_strategy']}"
create_folder(log_file)
log_file += '/data_en.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Dataset

In [10]:
print(f"{params['dataset']['path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}_resampled_{params['dataset']['sampling_size']}.json")

/workspaces/CodeSmells/datax/code_smells/generation/dataset/M1_q_none/greedy_search_resampled_500.json


In [12]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}_resampled_{params['dataset']['sampling_size']}.json", )

In [13]:
df_dataset

,id,commit_id,repo,path,file_name,commit_message,url,language,category,prompt,...,vocab_size,fun_name,complexity,nloc,token_counts,ast_errors,ast_levels,n_ast_nodes,n_ast_errors,n_identifiers
0,258876,1fc86b6aacd89da44a3b4e8abf7c3e2ba4336ffe,scikit-learn,sklearn/feature_selection/_univariate_selectio...,_univariate_selection.py,MNT Update black to stable version (#22474),https://github.com/scikit-learn/scikit-learn.git,Python,Convention,complete the following incomplete Python funct...,...,82,r_regression,8.0,29.0,271.0,[],15,434,0,31
1,172936,73d359af05701cb1cf2ace7dee26f14ea7238cb4,calibre-web,cps/gdriveutils.py,gdriveutils.py,Bugfix logging with gdrive\nUpdate optional-re...,https://github.com/janeczku/calibre-web.git,Python,Convention,complete the following incomplete Python funct...,...,38,getEbooksFolderId,3.0,14.0,83.0,[],14,296,0,21
2,176213,5dfd57af2a141a013ae3753e160180b82bec9469,networkx,networkx/tests/test_convert_scipy.py,test_convert_scipy.py,Use scipy.sparse array datastructure (#5139)\n...,https://github.com/networkx/networkx.git,Python,Convention,complete the following incomplete Python funct...,...,31,identity_conversion,1.0,22.0,271.0,[],10,414,0,22
3,258686,9f85c9d44965b764f40169ef2917e5f7a798684f,scikit-learn,sklearn/metrics/tests/test_pairwise.py,test_pairwise.py,TST Better info when checking for no warnings ...,https://github.com/scikit-learn/scikit-learn.git,Python,Convention,complete the following incomplete Python funct...,...,63,test_pairwise_boolean_distance,3.0,15.0,153.0,[],13,370,0,24
4,199358,06c66d6c69c6fe7955854a615956242d680f6b9a,sympy,sympy/physics/mechanics/tests/test_joint.py,test_joint.py,Add tests for intermediate frame and joint axi...,https://github.com/sympy/sympy.git,Python,Convention,complete the following incomplete Python funct...,...,60,test_pinjoint_axis,1.0,31.0,412.0,[],13,585,0,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24063,197310,65be461082dda54c8748922f9c29a19af1279fe1,sympy,sympy/core/basic.py,basic.py,Remove abbreviations in documentation,https://github.com/sympy/sympy.git,Python,Convention,complete the following incomplete Python funct...,...,92,replace._replace,4.0,6.0,56.0,[],20,752,0,32
24064,197310,65be461082dda54c8748922f9c29a19af1279fe1,sympy,sympy/core/basic.py,basic.py,Remove abbreviations in documentation,https://github.com/sympy/sympy.git,Python,Convention,complete the following incomplete Python funct...,...,92,replace._replace,4.0,6.0,56.0,[],20,752,0,32
24065,197310,65be461082dda54c8748922f9c29a19af1279fe1,sympy,sympy/core/basic.py,basic.py,Remove abbreviations in documentation,https://github.com/sympy/sympy.git,Python,Convention,complete the following incomplete Python funct...,...,92,replace._replace,4.0,6.0,56.0,[],20,752,0,32
24066,197310,65be461082dda54c8748922f9c29a19af1279fe1,sympy,sympy/core/basic.py,basic.py,Remove abbreviations in documentation,https://github.com/sympy/sympy.git,Python,Convention,complete the following incomplete Python funct...,...,92,replace._replace,4.0,6.0,56.0,[],20,752,0,32


#### Model Loading

In [14]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = CodeLlamaTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [15]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
model.config

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

#### preprocess dataset

In [17]:
df_dataset['input_ids'] = df_dataset[params['dataset']['content_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
df_dataset['input_lenght'] = df_dataset['input_ids'].map(lambda input_ids: len(input_ids))

#### Softmax Normalization and Data Engineering

In [18]:
def topk_tuple( logit_vocab_tensor, largest, tokenizer_fn):
    "Run topk for a token"
    topk = logit_vocab_tensor.topk( k=1 , largest=largest ) #TODO K number of elements can be extended
    return ( tokenizer.convert_tokens_to_string([tokenizer_fn.decode(topk.indices)]), topk.values.item())

def min_max_logits( logit_vocab_sample_tensor, tokenizer_fn ):
    "Compute min_max for a sample"
    max_cases = []
    min_cases = []
    for logit_vocab_tensor in logit_vocab_sample_tensor:
        max_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = True, tokenizer_fn = tokenizer_fn) ) #TST Max Logit
        min_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = False, tokenizer_fn = tokenizer_fn) ) #TST Min Logit
    return max_cases, min_cases

def actual_logit( 
                 logit_vocab_sample_tensor, 
                 tokenized_prompt, 
                 tokenizer_fn,
                 ):
    "Compute actual logits for a sample"
    actual_logits_prompt = []
    for token_pos, id_token in enumerate( tokenized_prompt[1:] ): #Eliminate the first token prediction since we do not use it
        actual_logits_prompt.append(
            (   tokenizer.convert_tokens_to_string([tokenizer_fn.decode( int(id_token))]), #retrieving the name of the token with the id
                logit_vocab_sample_tensor[token_pos][int(id_token)].item()) #retrieving the logit given the position in the sequence and the position in the vocab
            )
    return actual_logits_prompt

In [19]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [20]:
callbacks_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}"
out = np.load(f"{callbacks_dir}/logits_tensor[0]_batch[0].npy")

print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 530, 32016)


In [21]:
max_case,min_case = min_max_logits(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out], ####### 
    tokenizer_fn= tokenizer
    )
print(max_case)
assert len(max_case) == len(min_case)

[('<PRE>', 0.7492900490760803), ('module', 0.4794413447380066), ('pc', 0.05789488926529884), ('to', 0.03000357560813427), ('pr', 0.17619311809539795), ('(', 0.49110478162765503), ('x', 0.18413327634334564), (',', 0.7837546467781067), ('y', 0.4663568139076233), (',', 0.6742437481880188), ('l', 0.09025975316762924), ('args', 0.6018639206886292), ('l', 0.09314790368080139), ('=', 0.5727004408836365), ('True', 0.5564004182815552), (',', 0.8445126414299011), ('scale', 0.42541635036468506), ('_', 0.7570435404777527), ('center', 0.1138191744685173), ('=', 0.9234257340431213), ('True', 0.7369109988212585), (',', 0.6918704509735107), ('\n', 0.9534211158752441), ('  ', 0.8947504758834839), ('\n', 0.9500458836555481), ('  ', 0.962658166885376), ('"""', 0.1965550184249878), ('=', 0.5694356560707092), ('y', 0.9107635617256165), ('=', 0.8795614242553711), ('check', 0.45211222767829895), ('_', 0.9926853179931641), ('X', 0.7585020661354065), ('_', 0.9899470210075378), ('y', 0.9985018968582153), ('(', 

In [22]:
assert tokenizer.decode(df_dataset['input_ids'][0]) == df_dataset[params['dataset']['content_column']][0]
df_dataset[params['dataset']['content_column']][0]

2025-03-26 18:19:59.496239: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743013199.568880 1759697 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743013199.588815 1759697 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-26 18:19:59.735759: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


'def r_regression(X, y, *, center=True, force_finite=True):\n    \n    X, y = check_X_y(X, y, accept_sparse=["csr", "csc", "coo"], dtype=np.float64)\n    n_samples = X.shape[0]\n\n    # Compute centered values\n    # Note that E[(x - mean(x))*(y - mean(y))] = E[x*(y - mean(y))], so we\n    # need not center X\n    if center:\n        y = y - np.mean(y)\n        if issparse(X):\n            X_means = X.mean(axis=0).getA1()\n        else:\n            X_means = X.mean(axis=0)\n        # Compute the scaled standard deviations via moments\n        X_norms = np.sqrt(row_norms(X.T, squared=True) - n_samples * X_means*X_means)\n        X_norms[X_norms == 0.0] = 1.0\n        X = X / X_norms\n    else:\n        X_means = np.zeros(X.shape[1])\n\n    # Compute the design matrix\n    if issparse(X):\n        X_centered = X.copy()\n        X_centered.data -= X_means\n    else:\n        X_centered = X - X_means\n\n    # Compute the regression coefficients\n    if force_finite:\n        X_centered = 

In [23]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

actual_cases = actual_logit(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    tokenized_prompt = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
    )
actual_cases

[('def', 0.0007611316395923495),
 ('r', 0.0006275415071286261),
 ('_', 0.03887249529361725),
 ('re', 0.003169793402776122),
 ('gression', 0.06703148037195206),
 ('(', 0.49110478162765503),
 ('X', 0.06370403617620468),
 (',', 0.7837546467781067),
 ('y', 0.4663568139076233),
 (',', 0.6742437481880188),
 ('*', 0.010683641768991947),
 (',', 0.29887655377388),
 ('center', 0.00024064625904429704),
 ('=', 0.5727004408836365),
 ('True', 0.5564004182815552),
 (',', 0.8445126414299011),
 ('force', 0.0007708600023761392),
 ('_', 0.7570435404777527),
 ('finite', 0.002615149598568678),
 ('=', 0.9234257340431213),
 ('True', 0.7369109988212585),
 ('):', 0.3012572228908539),
 ('\n', 0.9534211158752441),
 ('   ', 0.007184024900197983),
 ('\n', 0.9500458836555481),
 ('  ', 0.962658166885376),
 ('X', 0.099087655544281),
 (',', 0.2046910971403122),
 ('y', 0.9107635617256165),
 ('=', 0.8795614242553711),
 ('check', 0.45211222767829895),
 ('_', 0.9926853179931641),
 ('X', 0.7585020661354065),
 ('_', 0.98994

#### Processing all the Batches

In [24]:
def batching_logits(tokenizer,tf_input_ids,size=10000):
    max_logit_token_prompt = []
    min_logit_token_prompt = []
    actual_logit_token_prompt = []

    
    soft = torch.nn.Softmax( dim = 0 )                          #Flattening normalization
    
    for file in range( size ):
        callbacks_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}"
        out = np.load(f"{callbacks_dir}/logits_tensor[{file}]_batch[{file}].npy") #<sample,tokens,voc_tokens>
        out = out[0]  ##### #<tokens,voc_tokens>
        next_tokens_distribution = [ soft( torch.from_numpy(token) ) for token in out]  #Flattening normalization
        
        max_cases,min_cases = min_max_logits(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenizer_fn= tokenizer
            )

        actual_cases = actual_logit(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenized_prompt = tf_input_ids[ file ],
            tokenizer_fn = tokenizer
            )
        
        max_logit_token_prompt.append( max_cases )
        min_logit_token_prompt.append( min_cases )
        actual_logit_token_prompt.append( actual_cases )
        
        logging.info(file)
    return max_logit_token_prompt,min_logit_token_prompt,actual_logit_token_prompt

In [25]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [68]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = batching_logits(
    tokenizer=tokenizer , tf_input_ids=input_ids_list, 
    size = len(df_dataset)
) #<---WARNING TIME Consuming

#### Saving results

In [69]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [70]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(5, 33)

In [ ]:
output_dir = f"{params['output_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}"
create_folder(output_dir)
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

#### Loss Retrieval

In [ ]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}"
    output_loss = []
    for current_batch in range(size):
        out = np.load(f"{output_dir}/_loss_batch[{current_batch}].npy")
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [76]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [77]:
output_loss

[0.6501871347427368,
 1.36393404006958,
 0.6265979409217834,
 0.9561377763748169,
 1.176944613456726]

In [78]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,input_lenght,input_ids,max_prob,min_prob,actual_prob,loss
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,_get_state_dict,Apply black formatting (#2115)\n\n* Testing bl...,def _get_state_dict(self):\n \n ...,https://github.com/deepset-ai/haystack.git,Python,...,2,22,state_dict = {,Warning,295,"[822, 903, 657, 29918, 3859, 29918, 8977, 2989...","[(<PRE>, 0.7492886185646057), (module, 0.47944...","[(<s>, 6.321602312800434e-13), ($}, 1.21012630...","[(def, 0.0007611338514834642), (_, 0.009010307...",0.650187
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,async_added_to_hass,Improve entity type hints [r] (#77874),async def async_added_to_hass(self) -> None:\n...,https://github.com/home-assistant/core.git,Python,...,4,37,self._dings_update_callback(),Warning,72,"[7465, 822, 7465, 29918, 23959, 29918, 517, 29...","[(<PRE>, 0.7492926716804504), (\n, 0.145589932...","[(<s>, 6.322408417115677e-13), (oreferrer, 3.7...","[(async, 1.0811768333951477e-05), (def, 0.0006...",1.363934
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,test_new_comment,Fix commenting thread notifications being sent...,def test_new_comment(self):\n post_data...,https://github.com/wagtail/wagtail.git,Python,...,33,125,self.assertEqual(mail.outbox[0].subjec...,Convention,665,"[822, 1243, 29918, 1482, 29918, 9342, 29898, 1...","[(<PRE>, 0.7492899298667908), (module, 0.47944...","[(<s>, 6.321468413832132e-13), ($}, 1.21012033...","[(def, 0.0007611322216689587), (test, 0.019638...",0.626598
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,__init__,Improve typehints / reduce warnings from mypy,"def __init__(self, **kwargs) -> None:\n ...",https://github.com/freqtrade/freqtrade.git,Python,...,13,44,import_str = 'stable_baselines3',Warning,493,"[822, 4770, 2344, 12035, 1311, 29892, 3579, 19...","[(<PRE>, 0.7492900490760803), (module, 0.47944...","[(<s>, 6.321457571810407e-13), ($}, 1.21011436...","[(def, 0.0007611316395923495), (__, 0.00598208...",0.956138
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,state,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,"def state(self, value):\n state_value =...",https://github.com/airbytehq/airbyte.git,Python,...,7,61,"self._start_date = max(self._state, se...",Warning,109,"[822, 2106, 29898, 1311, 29892, 995, 1125, 13,...","[(<PRE>, 0.749289870262146), (module, 0.479445...","[(<s>, 6.321889626376143e-13), ($}, 1.21010756...","[(def, 0.0007611394976265728), (state, 6.87381...",1.176945


In [79]:
## Saving CheckPoint 2
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

In [80]:
print("================================= PROCESS COMPLETED =================================")

================================= PROCESS COMPLETE =================================


In [82]:
del model
torch.cuda.empty_cache()
gc.collect()

0